In [1]:
CFG  = {
    "root-path":"/home/siddhesh/Documents/repos/llm-and-vlm-projects",
    "bitext-dataset-dir":"/data/raw/bitext",
    "multiwoz-dataset-dir":"/data/raw/multiwoz"
}

In [ ]:
from datasets import load_dataset

bitext = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset",cache_dir=CFG["root-path"]+CFG["bitext-dataset-dir"], split="train")
#multiwoz = load_dataset("Brendan/multiwoz_turns_v22_partitioned", cache_dir=CFG["root-path"]+CFG["multiwoz-dataset-dir"], split="train")


In [ ]:
print(raw)

In [4]:
from datasets import load_dataset

ds = load_dataset("pfb30/multi_woz_v22", trust_remote_code=True)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'pfb30/multi_woz_v22' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


RuntimeError: Dataset scripts are no longer supported, but found multi_woz_v22.py

In [9]:
from huggingface_hub import snapshot_download

path = snapshot_download(
    repo_id="pfb30/multi_woz_v22",
    repo_type="dataset",
    local_dir=CFG["root-path"]+CFG["multiwoz-dataset-dir"]
)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

In [19]:
import json
import requests
from datasets import Dataset, DatasetDict

# --- Build URL list (mirrors the .py script) ---
url_list = [
    ("dialogue_acts", "https://github.com/budzianowski/multiwoz/raw/master/data/MultiWOZ_2.2/dialog_acts.json")
]
url_list += [(f"train_{i:03d}", f"https://github.com/budzianowski/multiwoz/raw/master/data/MultiWOZ_2.2/train/dialogues_{i:03d}.json") for i in range(1, 18)]
url_list += [(f"dev_{i:03d}",   f"https://github.com/budzianowski/multiwoz/raw/master/data/MultiWOZ_2.2/dev/dialogues_{i:03d}.json")   for i in range(1, 3)]
url_list += [(f"test_{i:03d}",  f"https://github.com/budzianowski/multiwoz/raw/master/data/MultiWOZ_2.2/test/dialogues_{i:03d}.json")  for i in range(1, 3)]

# --- Download all files ---
print("Downloading files...")
data_files = {}
for name, url in url_list:
    print(f"  {name}...")
    r = requests.get(url)
    r.raise_for_status()
    data_files[name] = r.json()

dialogue_acts = data_files["dialogue_acts"]

# --- Process function (mirrors _generate_examples) ---
def process_split(split_prefix):
    records = []
    file_keys = [k for k in data_files if k.startswith(split_prefix)]
    for key in sorted(file_keys):
        for dialogue in data_files[key]:
            mapped_acts = dialogue_acts.get(dialogue["dialogue_id"], {})
            record = {
                "dialogue_id": dialogue["dialogue_id"],
                "services": dialogue["services"],
                "turns": [
                    {
                        "turn_id": turn["turn_id"],
                        "speaker": turn["speaker"],
                        "utterance": turn["utterance"],
                        "frames": [
                            {
                                "service": frame["service"],
                                "state": {
                                    "active_intent": frame["state"]["active_intent"] if "state" in frame else "",
                                    "requested_slots": frame["state"]["requested_slots"] if "state" in frame else [],
                                    "slots_values": {
                                        "slots_values_name": list(frame["state"]["slot_values"].keys()) if "state" in frame else [],
                                        "slots_values_list": list(frame["state"]["slot_values"].values()) if "state" in frame else [],
                                    },
                                },
                                "slots": [
                                    {
                                        "slot": slot["slot"],
                                        "value": "" if "copy_from" in slot else slot["value"],
                                        "start": slot.get("start", -1),
                                        "exclusive_end": slot.get("exclusive_end", -1),
                                        "copy_from": slot.get("copy_from", ""),
                                        "copy_from_value": slot["value"] if "copy_from" in slot else [],
                                    }
                                    for slot in frame["slots"]
                                ],
                            }
                            for frame in turn["frames"]
                        ],
                        "dialogue_acts": {
                            "dialog_act": [
                                {
                                    "act_type": act_type,
                                    "act_slots": {
                                        "slot_name": [s for s, _ in dialog_act],
                                        "slot_value": [v for _, v in dialog_act],
                                    },
                                }
                                for act_type, dialog_act in mapped_acts.get(turn["turn_id"], {}).get("dialog_act", {}).items()
                            ],
                            "span_info": [
                                {
                                    "act_type": s[0],
                                    "act_slot_name": s[1],
                                    "act_slot_value": s[2],
                                    "span_start": s[3],
                                    "span_end": s[4],
                                }
                                for s in mapped_acts.get(turn["turn_id"], {}).get("span_info", [])
                            ],
                        },
                    }
                    for turn in dialogue["turns"]
                ],
            }
            records.append(record)
    return records

# --- Build DatasetDict ---
print("Building dataset...")
ds = DatasetDict({
    "train":      Dataset.from_list(process_split("train")),
    "validation": Dataset.from_list(process_split("dev")),
    "test":       Dataset.from_list(process_split("test")),
})

print(ds)

  dialogue_acts...
  train_001...
  train_002...
  train_003...
  train_004...
  train_005...
  train_006...
  train_007...
  train_008...
  train_009...
  train_010...
  train_011...
  train_012...
  train_013...
  train_014...
  train_015...
  train_016...
  train_017...
  dev_001...
  dev_002...
  test_001...
  test_002...
Building dataset...
DatasetDict({
    train: Dataset({
        features: ['dialogue_id', 'services', 'turns'],
        num_rows: 8437
    })
    validation: Dataset({
        features: ['dialogue_id', 'services', 'turns'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['dialogue_id', 'services', 'turns'],
        num_rows: 1000
    })
})


In [20]:
ds.save_to_disk(CFG['root-path']+CFG['multiwoz-dataset-dir'])

Saving the dataset (0/1 shards):   0%|          | 0/8437 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]